# Comparison of ML models for football match prediction

## Description
This notebook contains a comprehensive comparison of ML models:
- **Result classification (H/D/A):** LR, RF, XGBoost, LightGBM, KNN, SVM, MLP, LSTM
- **Binary classification:** BTTS, Over 2.5
- **Goals regression:** MAE, RMSE, R²
- **Quality metrics:** Accuracy, Precision, Recall, F1, ROC/AUC, Cross-Validation
- **Technical metrics:** Training time, prediction time, RAM usage, CPU%, model size
- **Hyperparameter tuning:** GridSearchCV
- **Visualizations:** Confusion matrices, ROC curves, feature importance, scatter plots

**Data:** ~22,000 matches from 21+ leagues, domestic cups, European and international competitions  
**Features:** 113 features (form, table, H2H, momentum, fatigue, xG, player statistics)

In [ ]:
%pip install scikit-learn pandas numpy xgboost lightgbm psutil joblib matplotlib seaborn torch --quiet

In [ ]:
import json
import os
import time
import psutil
import joblib
import tempfile
import numpy as np
import pandas as pd
from datetime import datetime
from collections import defaultdict

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor

from sklearn.base import clone
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, auc,
    mean_absolute_error, mean_squared_error, r2_score
)

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import warnings
warnings.filterwarnings('ignore')

print(f"Analysis date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("Libraries loaded successfully!")

## 1. Loading data - all leagues and competitions

In [ ]:
DATA_DIR = r"c:\Users\tomas\Desktop\SofascoreDate\data"
COMP_TYPES = ['league', 'cups', 'european', 'international']

all_dfs = []
for comp_type in COMP_TYPES:
    comp_dir = os.path.join(DATA_DIR, comp_type)
    if not os.path.isdir(comp_dir):
        continue
    for country in os.listdir(comp_dir):
        country_dir = os.path.join(comp_dir, country)
        if not os.path.isdir(country_dir):
            continue
        for comp in os.listdir(country_dir):
            features_file = os.path.join(country_dir, comp, 'features', 'features_all_seasons.json')
            if not os.path.isfile(features_file):
                continue
            with open(features_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            samples = data.get('samples', [])
            if not samples:
                continue
            df_tmp = pd.DataFrame(samples)
            df_tmp['comp_type'] = comp_type
            df_tmp['country'] = country
            df_tmp['competition'] = comp
            if 'label_result_int' in df_tmp.columns:
                df_tmp = df_tmp[df_tmp['label_result_int'].notna()]
                if len(df_tmp) > 0:
                    all_dfs.append(df_tmp)
                    print(f"  {comp_type}/{country}/{comp}: {len(df_tmp)} matches")

df = pd.concat(all_dfs, ignore_index=True)
print(f"\n{'='*60}")
print(f"TOTAL NUMBER OF MATCHES: {len(df)}")
print(f"Number of competitions: {len(all_dfs)}")
print(f"Period: {df['date'].min()} - {df['date'].max()}")
print(f"Columns: {len(df.columns)}")

In [ ]:
FEATURE_COLUMNS = [
    'home_rest_days', 'away_rest_days', 'rest_days_diff',
    'home_is_congested', 'away_is_congested',
    'home_form_points', 'away_form_points', 'form_points_diff',
    'home_form_avg_points', 'away_form_avg_points',
    'home_form_wins', 'away_form_wins',
    'home_form_draws', 'away_form_draws',
    'home_form_losses', 'away_form_losses',
    'home_form_goals_for', 'away_form_goals_for',
    'home_form_goals_against', 'away_form_goals_against',
    'home_form_goal_diff', 'away_form_goal_diff',
    'home_form_xg_for', 'away_form_xg_for',
    'home_form_xg_against', 'away_form_xg_against',
    'home_form_xg_diff', 'away_form_xg_diff', 'xg_form_diff',
    'home_form_clean_sheets', 'away_form_clean_sheets',
    'home_form10_points', 'away_form10_points', 'form10_points_diff',
    'home_form10_avg_points', 'away_form10_avg_points',
    'home_form10_wins', 'away_form10_wins',
    'home_form10_goals_for', 'away_form10_goals_for',
    'home_form10_goals_against', 'away_form10_goals_against',
    'home_form10_goal_diff', 'away_form10_goal_diff',
    'home_form10_xg_for', 'away_form10_xg_for',
    'home_form10_xg_diff', 'away_form10_xg_diff',
    'home_form10_clean_sheets', 'away_form10_clean_sheets',
    'home_avg_player_rating', 'away_avg_player_rating', 'player_rating_diff',
    'home_top_scorer_goals', 'away_top_scorer_goals',
    'home_total_team_goals', 'away_total_team_goals',
    'home_total_team_assists', 'away_total_team_assists',
    'home_avg_minutes_starters', 'away_avg_minutes_starters',
    'home_squad_avg_age', 'away_squad_avg_age',
    'home_table_position', 'away_table_position', 'position_diff',
    'home_table_points', 'away_table_points', 'points_diff',
    'home_table_goal_diff', 'away_table_goal_diff',
    'home_table_ppg', 'away_table_ppg', 'ppg_diff',
    'h2h_matches', 'h2h_home_wins', 'h2h_away_wins', 'h2h_draws',
    'h2h_home_goals', 'h2h_away_goals', 'h2h_home_win_rate',
    'home_momentum_points', 'away_momentum_points', 'momentum_diff',
    'home_momentum_goals', 'away_momentum_goals',
    'home_momentum_xg', 'away_momentum_xg',
    'home_venue_form_points', 'away_venue_form_points', 'venue_ppg_diff',
    'home_venue_form_ppg', 'away_venue_form_ppg',
    'home_venue_form_goals_for', 'away_venue_form_goals_for',
    'home_venue_form_goals_against', 'away_venue_form_goals_against',
    'home_venue_form_clean_sheets', 'away_venue_form_clean_sheets',
    'home_fatigue_matches', 'away_fatigue_matches', 'fatigue_diff',
    'home_fatigue_avg_days', 'away_fatigue_avg_days',
    'home_sos_avg_position', 'away_sos_avg_position', 'sos_diff',
    'home_sos_avg_ppg', 'away_sos_avg_ppg',
    'home_clean_sheet_pct', 'away_clean_sheet_pct',
    'home_failed_to_score_pct', 'away_failed_to_score_pct',
]

available_features = [c for c in FEATURE_COLUMNS if c in df.columns]
print(f"Available features: {len(available_features)} / {len(FEATURE_COLUMNS)}")

label_cols = ['label_result_int', 'label_btts', 'label_over_2_5', 'label_home_goals', 'label_away_goals']
keep_cols = [c for c in available_features + label_cols if c in df.columns]
df_clean = df[keep_cols].copy().dropna(subset=['label_result_int'])
df_clean = df_clean.fillna(0)

X = df_clean[available_features].astype(float)
y_result = df_clean['label_result_int'].astype(int)
y_btts = df_clean['label_btts'].astype(int)
y_over25 = df_clean['label_over_2_5'].astype(int)
y_home_goals = df_clean['label_home_goals'].astype(float)
y_away_goals = df_clean['label_away_goals'].astype(float)

print(f"\nMatches after cleaning: {len(df_clean)}")
print(f"Features: {len(available_features)}")
print(f"\nResult distribution (H/D/A):")
for cls, name in {0: 'Home', 1: 'Draw', 2: 'Away'}.items():
    n = (y_result == cls).sum()
    print(f"  {name}: {n} ({n/len(y_result):.1%})")
print(f"\nBTTS: {y_btts.mean():.1%} | Over 2.5: {y_over25.mean():.1%}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_result, test_size=0.2, random_state=42, stratify=y_result
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=available_features, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=available_features, index=X_test.index)

# Splits for BTTS, Over 2.5, Regression (same index)
y_btts_train, y_btts_test = y_btts.loc[X_train.index], y_btts.loc[X_test.index]
y_o25_train, y_o25_test = y_over25.loc[X_train.index], y_over25.loc[X_test.index]
y_hg_train, y_hg_test = y_home_goals.loc[X_train.index], y_home_goals.loc[X_test.index]
y_ag_train, y_ag_test = y_away_goals.loc[X_train.index], y_away_goals.loc[X_test.index]

# X aliases for binary and regression tasks (same features, scaled)
X_btts_train, X_btts_test = X_train_scaled, X_test_scaled
X_o25_train, X_o25_test = X_train_scaled, X_test_scaled
X_reg_train, X_reg_test = X_train_scaled, X_test_scaled

print(f"Training set: {len(X_train)} matches")
print(f"Test set: {len(X_test)} matches")

## 2. Match result classification (H/D/A) - 7 tabular models

In [ ]:
def get_memory_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024 / 1024

def get_cpu_pct(interval=0.1):
    return psutil.Process(os.getpid()).cpu_percent(interval=interval)

def get_model_size_kb(model):
    with tempfile.NamedTemporaryFile(delete=False, suffix='.pkl') as f:
        path = f.name
    joblib.dump(model, path)
    size = os.path.getsize(path) / 1024
    os.unlink(path)
    return size

def train_and_evaluate(model, X_tr, X_te, y_tr, y_te, name, use_scaling=False):
    Xtr = X_train_scaled if use_scaling else X_tr
    Xte = X_test_scaled if use_scaling else X_te

    mem_before = get_memory_mb()
    _ = get_cpu_pct(0.01)  # warm up
    t0 = time.time()
    model.fit(Xtr, y_tr)
    train_time = time.time() - t0
    cpu_train = get_cpu_pct(0.01)
    mem_after = get_memory_mb()

    t0 = time.time()
    y_pred = model.predict(Xte)
    predict_time = (time.time() - t0) * 1000  # ms

    y_proba = model.predict_proba(Xte) if hasattr(model, 'predict_proba') else None

    cv = cross_val_score(model, Xtr, y_tr, cv=5, scoring='accuracy', n_jobs=-1)

    return {
        'name': name,
        'accuracy': accuracy_score(y_te, y_pred),
        'precision': precision_score(y_te, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_te, y_pred, average='weighted', zero_division=0),
        'f1': f1_score(y_te, y_pred, average='weighted', zero_division=0),
        'cv_mean': cv.mean(), 'cv_std': cv.std(),
        'train_time': train_time,
        'predict_time': predict_time / 1000,  # seconds
        'memory_mb': max(0.1, mem_after - mem_before),
        'cpu_pct': cpu_train,
        'model_size_mb': get_model_size_kb(model) / 1024,
        'y_pred': y_pred, 'y_proba': y_proba, 'model': model,
    }

print("Evaluation functions defined.")

In [ ]:
models_config = {
    'Logistic Regression': {
        'model': LogisticRegression(max_iter=1000, multi_class='multinomial', solver='lbfgs', random_state=42),
        'scaling': True,
    },
    'Random Forest': {
        'model': RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_split=10,
            min_samples_leaf=5, max_features='sqrt', random_state=42, n_jobs=-1, class_weight='balanced'),
        'scaling': False,
    },
    'XGBoost': {
        'model': XGBClassifier(n_estimators=200, max_depth=8, learning_rate=0.1, subsample=0.8,
            colsample_bytree=0.8, random_state=42, n_jobs=-1, eval_metric='mlogloss'),
        'scaling': False,
    },
    'LightGBM': {
        'model': LGBMClassifier(n_estimators=200, max_depth=8, learning_rate=0.1, subsample=0.8,
            colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1),
        'scaling': False,
    },
    'KNN': {
        'model': KNeighborsClassifier(n_neighbors=15, weights='distance', metric='minkowski', n_jobs=-1),
        'scaling': True,
    },
    'SVM': {
        'model': SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42, class_weight='balanced'),
        'scaling': True,
    },
    'MLP': {
        'model': MLPClassifier(hidden_layer_sizes=(128, 64, 32), activation='relu', solver='adam',
            alpha=0.001, max_iter=500, random_state=42, early_stopping=True, validation_fraction=0.1),
        'scaling': True,
    },
}

print(f"Defined {len(models_config)} models for comparison:")
for name in models_config:
    print(f"  - {name}")

In [ ]:
print("="*70)
print("MODEL TRAINING AND EVALUATION")
print("="*70)

all_results = {}
for name, cfg in models_config.items():
    print(f"\nTraining: {name}...")
    res = train_and_evaluate(cfg['model'], X_train, X_test, y_train, y_test, name, cfg['scaling'])
    all_results[name] = res
    print(f"  Accuracy: {res['accuracy']:.1%} | F1: {res['f1']:.3f} | Time: {res['train_time']:.2f}s | Model: {res['model_size_mb']:.2f} MB")

print("\n" + "="*70)
print("All models trained!")

In [ ]:
cols_quality = ['accuracy', 'precision', 'recall', 'f1', 'cv_mean', 'cv_std']
cols_tech = ['train_time', 'predict_time', 'memory_mb', 'cpu_pct', 'model_size_mb']

df_q = pd.DataFrame({n: {k: all_results[n][k] for k in cols_quality} for n in all_results}).T
df_t = pd.DataFrame({n: {k: all_results[n][k] for k in cols_tech} for n in all_results}).T

print("="*80)
print("QUALITY METRICS")
print("="*80)
print(df_q.round(4).to_string())

print("\n" + "="*80)
print("TECHNICAL METRICS")
print("="*80)
print(df_t.round(3).to_string())

print(f"\nBest model (accuracy): {df_q['accuracy'].idxmax()} ({df_q['accuracy'].max():.1%})")
print(f"Best model (F1): {df_q['f1'].idxmax()} ({df_q['f1'].max():.3f})")
print(f"Fastest training: {df_t['train_time'].idxmin()} ({df_t['train_time'].min():.2f}s)")
print(f"Smallest model: {df_t['model_size_mb'].idxmin()} ({df_t['model_size_mb'].min():.2f} MB)")

In [ ]:
n_models = len(all_results)
n_cols = 4
n_rows = (n_models + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()
class_names = ['Home', 'Draw', 'Away']

for i, (name, res) in enumerate(all_results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=class_names, yticklabels=class_names)
    axes[i].set_title(f"{name}\nAcc: {res['accuracy']:.1%}", fontsize=10)
    axes[i].set_ylabel('Actual')
    axes[i].set_xlabel('Prediction')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Confusion matrices - Classification H/D/A', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
n_classes = 3
class_labels = ['Home', 'Draw', 'Away']
colors_roc = ['#e74c3c', '#f39c12', '#2ecc71']

models_with_proba = [(n, r) for n, r in all_results.items() if r['y_proba'] is not None]
n_m = len(models_with_proba)
n_cols_roc = 4
n_rows_roc = (n_m + n_cols_roc - 1) // n_cols_roc
fig, axes = plt.subplots(n_rows_roc, n_cols_roc, figsize=(16, 4 * n_rows_roc))
axes = axes.flatten()

auc_scores = {}
for idx, (name, res) in enumerate(models_with_proba):
    ax = axes[idx]
    model_aucs = []
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], res['y_proba'][:, i])
        roc_auc = auc(fpr, tpr)
        model_aucs.append(roc_auc)
        ax.plot(fpr, tpr, color=colors_roc[i], lw=2, label=f'{class_labels[i]} (AUC={roc_auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    macro_auc = np.mean(model_aucs)
    ax.set_title(f'{name}\nMacro AUC: {macro_auc:.3f}', fontsize=10)
    ax.legend(fontsize=7, loc='lower right')
    auc_scores[name] = macro_auc

for j in range(idx + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('ROC curves - One-vs-Rest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nMacro AUC per model:")
for name, score in sorted(auc_scores.items(), key=lambda x: -x[1]):
    print(f"  {name}: {score:.4f}")

In [ ]:
print("="*80)
print("DETAILED CLASSIFICATION REPORTS")
print("="*80)

for name, res in all_results.items():
    print(f"\n{'-'*40}")
    print(f"{name}")
    print(f"{'-'*40}")
    print(classification_report(y_test, res['y_pred'], target_names=['Home', 'Draw', 'Away']))

## 3. LSTM Model (Long Short-Term Memory)
The LSTM recurrent network processes match features as a sequence: home and away team features are treated as two time steps, allowing the model to capture relationships between both teams.


In [ ]:
# Grouping features into temporal blocks:
# Step 0: form10 features (long-term - last 10 matches)
# Step 1: form5 features (short-term - last 5 matches)
# Step 2: Current features (table, H2H, momentum, fatigue, SOS, patterns)
# This allows the LSTM to see progression: long-term -> short-term -> current

home_form10 = [c for c in available_features if 'home_form10' in c]
away_form10 = [c for c in available_features if 'away_form10' in c]
home_form5 = [c for c in available_features if 'home_form_' in c and 'form10' not in c]
away_form5 = [c for c in available_features if 'away_form_' in c and 'form10' not in c]
other_feats = [c for c in available_features if c not in home_form10 + away_form10 + home_form5 + away_form5]

max_block = max(len(home_form10 + away_form10), len(home_form5 + away_form5), len(other_feats))

def make_block(df_in, cols, max_len):
    arr = df_in[cols].values if cols else np.zeros((len(df_in), 0))
    pad_width = max_len - arr.shape[1]
    if pad_width > 0:
        arr = np.hstack([arr, np.zeros((arr.shape[0], pad_width))])
    return arr

X_train_np = scaler.fit_transform(X_train)
X_test_np = scaler.transform(X_test)
X_train_df_s = pd.DataFrame(X_train_np, columns=available_features, index=X_train.index)
X_test_df_s = pd.DataFrame(X_test_np, columns=available_features, index=X_test.index)

def build_sequences(df_in):
    b0 = make_block(df_in, home_form10 + away_form10, max_block)
    b1 = make_block(df_in, home_form5 + away_form5, max_block)
    b2 = make_block(df_in, other_feats, max_block)
    return np.stack([b0, b1, b2], axis=1)  # (N, 3, max_block)

X_seq_train = build_sequences(X_train_df_s)
X_seq_test = build_sequences(X_test_df_s)

print(f"Training sequence shape: {X_seq_train.shape}  (samples, time_steps, features)")
print(f"Test sequence shape:    {X_seq_test.shape}")

In [ ]:
class FootballLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, num_classes=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        out, (h_n, _) = self.lstm(x)
        # Concatenate last hidden states from both directions
        h_cat = torch.cat((h_n[-2], h_n[-1]), dim=1)
        return self.fc(h_cat)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

input_size = X_seq_train.shape[2]
lstm_model = FootballLSTM(input_size=input_size, hidden_size=64, num_layers=2, num_classes=3).to(device)

train_ds = TensorDataset(
    torch.FloatTensor(X_seq_train),
    torch.LongTensor(y_train.values)
)
test_ds = TensorDataset(
    torch.FloatTensor(X_seq_test),
    torch.LongTensor(y_test.values)
)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=512)

optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

epochs = 50
best_val_acc = 0
patience_counter = 0

mem_before = get_memory_mb()
_ = get_cpu_pct(0.01)
t0 = time.time()

for epoch in range(epochs):
    lstm_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = lstm_model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    lstm_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = lstm_model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += len(yb)
    val_acc = correct / total
    scheduler.step(total_loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        best_state = lstm_model.state_dict().copy()
    else:
        patience_counter += 1
        if patience_counter >= 10:
            break

    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1}/{epochs}: loss={total_loss/len(train_loader):.4f}, val_acc={val_acc:.1%}")

lstm_train_time = time.time() - t0
lstm_cpu = get_cpu_pct(0.01)
lstm_mem = max(0.1, get_memory_mb() - mem_before)

lstm_model.load_state_dict(best_state)
print(f"\nLSTM - best val accuracy: {best_val_acc:.1%} (training time: {lstm_train_time:.1f}s)")

In [ ]:
lstm_model.eval()
all_preds, all_true, all_probs = [], [], []
t0_pred = time.time()
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = lstm_model(xb)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(yb.numpy())
        all_probs.extend(probs)

lstm_predict_time = time.time() - t0_pred
all_preds = np.array(all_preds)
all_true = np.array(all_true)
all_probs = np.array(all_probs)

lstm_acc = accuracy_score(all_true, all_preds)
lstm_prec = precision_score(all_true, all_preds, average='weighted', zero_division=0)
lstm_rec = recall_score(all_true, all_preds, average='weighted', zero_division=0)
lstm_f1 = f1_score(all_true, all_preds, average='weighted', zero_division=0)

tmp_path = "tmp_lstm.pt"
torch.save(best_state, tmp_path)
lstm_size = os.path.getsize(tmp_path) / (1024 * 1024)
os.remove(tmp_path)

all_results['LSTM'] = {
    'accuracy': lstm_acc,
    'precision': lstm_prec,
    'recall': lstm_rec,
    'f1': lstm_f1,
    'cv_mean': best_val_acc,   # use val acc as CV proxy
    'cv_std': 0.0,
    'train_time': lstm_train_time,
    'predict_time': lstm_predict_time,
    'memory_mb': lstm_mem,
    'cpu_pct': lstm_cpu,
    'model_size_mb': lstm_size,
    'y_pred': all_preds,
    'y_proba': all_probs,
}

print(f"LSTM: accuracy={lstm_acc:.1%}, precision={lstm_prec:.1%}, recall={lstm_rec:.1%}, F1={lstm_f1:.1%}")
print(f"  Training time: {lstm_train_time:.1f}s, prediction: {lstm_predict_time:.3f}s, size: {lstm_size:.2f} MB")

## 4. Hyperparameter tuning (GridSearchCV)

Comparison of default vs. optimized hyperparameters for the top 3 models.

In [ ]:
param_grids = {
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42, n_jobs=-1),
        'params': {
            'n_estimators': [100, 300, 500],
            'max_depth': [10, 20, None],
            'min_samples_split': [2, 5],
        }
    },
    'XGBoost': {
        'model': XGBClassifier(
            random_state=42, eval_metric='mlogloss',
            tree_method='hist', n_jobs=-1, verbosity=0
        ),
        'params': {
            'n_estimators': [100, 300, 500],
            'max_depth': [4, 6, 8],
            'learning_rate': [0.01, 0.05, 0.1],
        }
    },
    'LightGBM': {
        'model': LGBMClassifier(random_state=42, verbosity=-1, n_jobs=-1),
        'params': {
            'n_estimators': [100, 300, 500],
            'max_depth': [5, 10, -1],
            'learning_rate': [0.01, 0.05, 0.1],
        }
    },
}

tuning_results = {}
for name, cfg in param_grids.items():
    print(f"Tuning {name}...")
    gs = GridSearchCV(cfg['model'], cfg['params'], cv=3, scoring='accuracy',
                      n_jobs=-1, verbose=0, refit=True)
    gs.fit(X_train_scaled, y_train)

    best_model = gs.best_estimator_
    y_pred_tuned = best_model.predict(X_test_scaled)

    tuning_results[name] = {
        'best_params': gs.best_params_,
        'cv_score': gs.best_score_,
        'test_accuracy_default': all_results[name]['accuracy'],
        'test_accuracy_tuned': accuracy_score(y_test, y_pred_tuned),
        'test_f1_tuned': f1_score(y_test, y_pred_tuned, average='weighted', zero_division=0),
    }
    print(f"  Best parameters: {gs.best_params_}")
    print(f"  Accuracy: {all_results[name]['accuracy']:.1%} -> {tuning_results[name]['test_accuracy_tuned']:.1%}")

tuning_df = pd.DataFrame(tuning_results).T
tuning_df = tuning_df[['test_accuracy_default', 'test_accuracy_tuned', 'test_f1_tuned', 'cv_score']]
tuning_df.columns = ['Accuracy (default)', 'Accuracy (tuned)', 'F1 (tuned)', 'CV Score']
for col in tuning_df.columns:
    tuning_df[col] = tuning_df[col].map(lambda x: f"{x:.1%}")
print("\n=== Tuning comparison ===")
display(tuning_df)

## 5. Binary classification: BTTS and Over 2.5

Using the same models for prediction:
- **BTTS** (Both Teams To Score) - will both teams score
- **Over 2.5** - will the total be more than 2.5 goals

In [ ]:
binary_tasks = {
    'BTTS': (X_btts_train, X_btts_test, y_btts_train, y_btts_test),
    'Over 2.5': (X_o25_train, X_o25_test, y_o25_train, y_o25_test),
}

binary_models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42, n_jobs=-1),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=300, eval_metric='logloss',
                             tree_method='hist', random_state=42, verbosity=0),
    'LightGBM': LGBMClassifier(n_estimators=300, random_state=42, verbosity=-1, n_jobs=-1),
}

binary_results = {}
for task_name, (Xtr, Xte, ytr, yte) in binary_tasks.items():
    print(f"\n{'='*50}")
    print(f"  {task_name}")
    print(f"{'='*50}")
    print(f"  Distribution: train={dict(pd.Series(ytr).value_counts())}, test={dict(pd.Series(yte).value_counts())}")

    task_res = {}
    for model_name, model in binary_models.items():
        m = clone(model)
        m.fit(Xtr, ytr)
        y_pred = m.predict(Xte)
        y_prob = m.predict_proba(Xte)[:, 1] if hasattr(m, 'predict_proba') else None

        acc = accuracy_score(yte, y_pred)
        prec = precision_score(yte, y_pred, zero_division=0)
        rec = recall_score(yte, y_pred, zero_division=0)
        f1 = f1_score(yte, y_pred, zero_division=0)
        auc_val = roc_auc_score(yte, y_prob) if y_prob is not None else 0.0

        task_res[model_name] = {
            'accuracy': acc, 'precision': prec, 'recall': rec,
            'f1': f1, 'auc': auc_val
        }
        print(f"  {model_name}: acc={acc:.1%}, prec={prec:.1%}, rec={rec:.1%}, F1={f1:.1%}, AUC={auc_val:.3f}")
    binary_results[task_name] = task_res

for task_name, task_res in binary_results.items():
    df = pd.DataFrame(task_res).T
    for col in df.columns:
        df[col] = df[col].map(lambda x: f"{x:.1%}" if col != 'auc' else f"{x:.3f}")
    print(f"\n=== {task_name} - Model comparison ===")
    display(df)

## 6. Regression: Goal count prediction

Regression models predicting the number of home and away goals. Metrics: MAE, RMSE, R².

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

regression_targets = {
    'Home goals': (y_hg_train, y_hg_test),
    'Away goals': (y_ag_train, y_ag_test),
}

regression_models = {
    'Random Forest': RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=300, tree_method='hist', random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=300, random_state=42, verbosity=-1, n_jobs=-1),
}

regression_results = {}
for target_name, (ytr, yte) in regression_targets.items():
    print(f"\n{'='*50}")
    print(f"  {target_name}")
    print(f"{'='*50}")
    print(f"  Average: train={ytr.mean():.2f}, test={yte.mean():.2f}")

    target_res = {}
    for model_name, model in regression_models.items():
        m = clone(model)
        # Use the same X as for classification (scaled)
        Xtr = X_reg_train
        Xte = X_reg_test
        m.fit(Xtr, ytr)
        y_pred = m.predict(Xte)

        mae = mean_absolute_error(yte, y_pred)
        rmse = np.sqrt(mean_squared_error(yte, y_pred))
        r2 = r2_score(yte, y_pred)

        target_res[model_name] = {'MAE': mae, 'RMSE': rmse, 'R²': r2}
        print(f"  {model_name}: MAE={mae:.3f}, RMSE={rmse:.3f}, R²={r2:.3f}")
    regression_results[target_name] = target_res

for target_name, target_res in regression_results.items():
    df = pd.DataFrame(target_res).T
    df['MAE'] = df['MAE'].map(lambda x: f"{x:.3f}")
    df['RMSE'] = df['RMSE'].map(lambda x: f"{x:.3f}")
    df['R²'] = df['R²'].map(lambda x: f"{x:.3f}")
    print(f"\n=== {target_name} - Regression comparison ===")
    display(df)

## 7. Feature Importance

Analysis of the most important features based on Random Forest and XGBoost.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 10))

for ax, model_name in zip(axes, ['Random Forest', 'XGBoost']):
    if model_name == 'Random Forest':
        m = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    else:
        m = XGBClassifier(n_estimators=300, eval_metric='mlogloss',
                          tree_method='hist', random_state=42, verbosity=0)
    m.fit(X_train_scaled, y_train)
    importances = m.feature_importances_

    fi = pd.Series(importances, index=FEATURE_COLUMNS).sort_values(ascending=True)
    top20 = fi.tail(20)

    top20.plot(kind='barh', ax=ax, color='steelblue', edgecolor='navy')
    ax.set_title(f'{model_name} - Top 20 features', fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: feature_importance.png")

## 8. Impact of data size on model quality

How model accuracy changes depending on the number of training samples.

In [ ]:
data_sizes = [500, 1000, 2000, 5000, 10000, len(X_train_scaled)]
test_models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42, n_jobs=-1),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=200, eval_metric='mlogloss',
                             tree_method='hist', random_state=42, verbosity=0),
    'LightGBM': LGBMClassifier(n_estimators=200, random_state=42, verbosity=-1, n_jobs=-1),
}

size_results = {name: [] for name in test_models}
actual_sizes = []

for n in data_sizes:
    n = min(n, len(X_train_scaled))
    if n not in actual_sizes:
        actual_sizes.append(n)
    else:
        continue

    idx = np.random.RandomState(42).choice(len(X_train_scaled), n, replace=False)
    X_sub = X_train_scaled[idx]
    y_sub = y_train.iloc[idx]

    for name, model in test_models.items():
        m = clone(model)
        m.fit(X_sub, y_sub)
        acc = accuracy_score(y_test, m.predict(X_test_scaled))
        size_results[name].append(acc)

fig, ax = plt.subplots(figsize=(12, 6))
markers = ['o', 's', '^', 'D']
for (name, accs), marker in zip(size_results.items(), markers):
    ax.plot(actual_sizes, accs, marker=marker, linewidth=2, markersize=8, label=name)

ax.set_xlabel('Number of training samples', fontsize=12)
ax.set_ylabel('Accuracy on test set', fontsize=12)
ax.set_title('Impact of data size on model quality', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xscale('log')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('data_size_impact.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: data_size_impact.png")

## 9. Comparative visualizations

Comprehensive charts comparing all models in terms of quality and computational costs.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

model_names = list(all_results.keys())
accs = [all_results[n]['accuracy'] for n in model_names]
train_times = [all_results[n]['train_time'] for n in model_names]
f1s = [all_results[n]['f1'] for n in model_names]
sizes = [all_results[n]['model_size_mb'] for n in model_names]
mems = [all_results[n]['memory_mb'] for n in model_names]

colors = plt.cm.tab10(np.linspace(0, 1, len(model_names)))

ax = axes[0, 0]
for i, name in enumerate(model_names):
    ax.scatter(train_times[i], accs[i], s=150, c=[colors[i]], edgecolors='black',
               zorder=5, label=name)
ax.set_xlabel('Training time (s)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Accuracy vs Training time', fontsize=13, fontweight='bold')
ax.legend(fontsize=8, loc='lower right')
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
bars = ax.barh(model_names, f1s, color=colors, edgecolor='black')
for bar, val in zip(bars, f1s):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.1%}', va='center', fontsize=10)
ax.set_xlabel('F1 Score (weighted)', fontsize=12)
ax.set_title('F1 Score - comparison', fontsize=13, fontweight='bold')
ax.set_xlim(0, max(f1s) * 1.15)
ax.grid(True, alpha=0.3, axis='x')

ax = axes[1, 0]
bars = ax.barh(model_names, sizes, color=colors, edgecolor='black')
for bar, val in zip(bars, sizes):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.2f} MB', va='center', fontsize=10)
ax.set_xlabel('Rozmiar modelu (MB)', fontsize=12)
ax.set_title('Rozmiar modelu - comparison', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

cv_means = [all_results[n]['cv_mean'] for n in model_names]
cv_stds = [all_results[n]['cv_std'] for n in model_names]
ax = axes[1, 1]
ax.barh(model_names, cv_means, xerr=cv_stds, color=colors, edgecolor='black',
        capsize=5, ecolor='gray')
for i, (val, std) in enumerate(zip(cv_means, cv_stds)):
    ax.text(val + std + 0.005, i, f'{val:.1%} ± {std:.1%}', va='center', fontsize=10)
ax.set_xlabel('CV Accuracy (5-fold)', fontsize=12)
ax.set_title('Cross-Validation - comparison', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('model_comparison_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: model_comparison_charts.png")

In [ ]:
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1', 'CV Mean',
                'Training time (s)', 'Memory (MB)', 'Size (MB)']
raw_data = []
for name in model_names:
    r = all_results[name]
    raw_data.append([r['accuracy'], r['precision'], r['recall'], r['f1'],
                     r['cv_mean'], r['train_time'], r['memory_mb'], r['model_size_mb']])

raw_df = pd.DataFrame(raw_data, index=model_names, columns=metric_names)

# Normalize 0-1 (for less=better metrics, invert)
norm_df = raw_df.copy()
for col in metric_names:
    vals = norm_df[col]
    if col in ['Training time (s)', 'Memory (MB)', 'Size (MB)']:
        # Less = better -> invert
        norm_df[col] = 1 - (vals - vals.min()) / (vals.max() - vals.min() + 1e-9)
    else:
        norm_df[col] = (vals - vals.min()) / (vals.max() - vals.min() + 1e-9)

fig, ax = plt.subplots(figsize=(14, 7))
im = ax.imshow(norm_df.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(len(metric_names)))
ax.set_xticklabels(metric_names, rotation=35, ha='right', fontsize=11)
ax.set_yticks(range(len(model_names)))
ax.set_yticklabels(model_names, fontsize=11)

for i in range(len(model_names)):
    for j in range(len(metric_names)):
        raw_val = raw_df.iloc[i, j]
        txt = f'{raw_val:.1%}' if j < 5 else f'{raw_val:.2f}'
        ax.text(j, i, txt, ha='center', va='center', fontsize=9,
                color='black' if norm_df.iloc[i, j] > 0.3 else 'white')

plt.colorbar(im, ax=ax, label='Normalized score (1=best)')
ax.set_title('Model comparison heatmap (all metrics)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('model_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: model_heatmap.png")

## 10. Summary and conclusions

Export results to CSV + final statistics.

In [ ]:
print("=" * 70)
print("  ML MODEL COMPARISON SUMMARY")
print("=" * 70)

summary_cols = ['accuracy', 'precision', 'recall', 'f1', 'cv_mean', 'cv_std',
                'train_time', 'predict_time', 'memory_mb', 'cpu_pct', 'model_size_mb']
summary_data = {}
for name in all_results:
    summary_data[name] = {col: all_results[name][col] for col in summary_cols}

summary_df = pd.DataFrame(summary_data).T.sort_values('accuracy', ascending=False)

print(f"\nNumber of models: {len(summary_df)}")
print(f"Data count: {len(X_train_scaled) + len(X_test_scaled):,} matches")
print(f"Number of features: {len(FEATURE_COLUMNS)}")
print(f"Split: {len(X_train_scaled):,} train / {len(X_test_scaled):,} test\n")

print("--- RANKING by Accuracy ---")
for i, (name, row) in enumerate(summary_df.iterrows(), 1):
    print(f"  {i}. {name:25s} -> {row['accuracy']:.1%} (F1: {row['f1']:.1%}, CV: {row['cv_mean']:.1%} ± {row['cv_std']:.1%})")

print(f"\n--- Fastest training: {summary_df['train_time'].idxmin()} ({summary_df['train_time'].min():.2f}s)")
print(f"--- Fastest prediction: {summary_df['predict_time'].idxmin()} ({summary_df['predict_time'].min():.4f}s)")
print(f"--- Smallest model: {summary_df['model_size_mb'].idxmin()} ({summary_df['model_size_mb'].min():.2f} MB)")
print(f"--- Best accuracy: {summary_df.index[0]} ({summary_df['accuracy'].iloc[0]:.1%})")
print(f"--- Best F1: {summary_df['f1'].idxmax()} ({summary_df['f1'].max():.1%})")

summary_export = summary_df.copy()
summary_export.to_csv('model_comparison_results.csv', float_format='%.4f')
print(f"\nResults saved to: model_comparison_results.csv")

display(summary_df.style.format({
    'accuracy': '{:.1%}', 'precision': '{:.1%}', 'recall': '{:.1%}',
    'f1': '{:.1%}', 'cv_mean': '{:.1%}', 'cv_std': '{:.1%}',
    'train_time': '{:.2f}s', 'predict_time': '{:.4f}s',
    'memory_mb': '{:.1f} MB', 'cpu_pct': '{:.1f}%', 'model_size_mb': '{:.2f} MB'
}).background_gradient(subset=['accuracy', 'f1'], cmap='Greens'))